In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.optimizers import Adam


In [2]:
def load_path(path):
    dataset = []
    for folder in os.listdir(path):
        folder = path + '/' + str(folder)
        if os.path.isdir(folder):
            for body in os.listdir(folder):
                path_p = folder + '/' + str(body)
                for id_p in os.listdir(path_p):
                    path_id = path_p + '/' + str(id_p)
                    for lab in os.listdir(path_id):

                        # ADD THIS LINE
                        fracture_label = "fractured" if "positive" in lab else "normal"

                        path_l = path_id + '/' + str(lab)
                        for img in os.listdir(path_l):
                            img_path = path_l + '/' + str(img)
                            dataset.append(
                                {
                                    "body_label": body,                 # Elbow / Hand / Shoulder
                                    "fracture_label": fracture_label,   # fractured / normal
                                    "image_path": img_path
                                }
                            )
    return dataset


In [4]:
import os

try:
    THIS_FOLDER = os.path.dirname(os.path.abspath(__file__))
except NameError:
    THIS_FOLDER = os.getcwd()

image_dir = os.path.join(THIS_FOLDER, "Dataset")

data = load_path(image_dir)

body_labels = []
fracture_labels = []
filepaths = []

for row in data:
    body_labels.append(row["body_label"])
    fracture_labels.append(row["fracture_label"])
    filepaths.append(row["image_path"])

df = pd.DataFrame({
    "Filepath": filepaths,
    "BodyLabel": body_labels,
    "FractureLabel": fracture_labels
})


In [5]:
train_df, test_df = train_test_split(
    df, train_size=0.9, shuffle=True, random_state=1
)


In [7]:
train_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2
)
test_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

In [11]:
train_images_body = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col="Filepath",
    y_col="BodyLabel",
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=64,
    shuffle=True,
    seed=42,
    subset="training"
)

val_images_body = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col="Filepath",
    y_col="BodyLabel",
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=64,
    shuffle=True,
    seed=42,
    subset="validation"
)
test_images = test_generator.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='BodyLabel',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=False
)


Found 14641 validated image filenames belonging to 3 classes.
Found 3660 validated image filenames belonging to 3 classes.
Found 2034 validated image filenames belonging to 3 classes.


In [12]:
train_images_frac = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col="Filepath",
    y_col="FractureLabel",
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=64,
    shuffle=True,
    seed=42,
    subset="training"
)

val_images_frac = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col="Filepath",
    y_col="FractureLabel",
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=64,
    shuffle=True,
    seed=42,
    subset="validation"
)
test_images = test_generator.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='FractureLabel',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=False
)

Found 14641 validated image filenames belonging to 2 classes.
Found 3660 validated image filenames belonging to 2 classes.
Found 2034 validated image filenames belonging to 2 classes.


In [13]:
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)
base_model.trainable = False


In [14]:
inputs = base_model.input
x = tf.keras.layers.Dense(128, activation="relu")(base_model.output)
x = tf.keras.layers.Dense(64, activation="relu")(x)

body_output = tf.keras.layers.Dense(3, activation="softmax")(x)

body_model = tf.keras.Model(inputs, body_output)

body_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
callbacks = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
body_model.fit(
    train_images_body,
    validation_data=val_images_body,
    epochs=5,
    callbacks=[callbacks]
)
body_model.save("weights/EfficientNetB1_BodyPart_Final.h5")

Epoch 1/5


c:\Users\Uday p\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


229/229 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8073 - loss: 0.5264

KeyboardInterrupt: 

In [15]:
inputs = base_model.input
x = tf.keras.layers.Dense(128, activation="relu")(base_model.output)
x = tf.keras.layers.Dense(64, activation="relu")(x)

fracture_output = tf.keras.layers.Dense(2, activation="softmax")(x)

fracture_model = tf.keras.Model(inputs, fracture_output)

fracture_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history=fracture_model.fit(
    train_images_frac,
    validation_data=val_images_frac,
    epochs=5,
    callbacks=[callbacks]
)

fracture_model.save("weights/EfficientNetB1_Fracture.h5")


Epoch 1/5


c:\Users\Uday p\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


229/229 ━━━━━━━━━━━━━━━━━━━━ 423s 2s/step - accuracy: 0.6316 - loss: 0.6415 - val_accuracy: 0.7090 - val_loss: 0.5834
Epoch 2/5
229/229 ━━━━━━━━━━━━━━━━━━━━ 335s 1s/step - accuracy: 0.7123 - loss: 0.5748 - val_accuracy: 0.7251 - val_loss: 0.5530
Epoch 3/5
229/229 ━━━━━━━━━━━━━━━━━━━━ 333s 1s/step - accuracy: 0.7311 - loss: 0.5462 - val_accuracy: 0.7238 - val_loss: 0.5521
Epoch 4/5
229/229 ━━━━━━━━━━━━━━━━━━━━ 348s 2s/step - accuracy: 0.7487 - loss: 0.5287 - val_accuracy: 0.7489 - val_loss: 0.5304
Epoch 5/5
229/229 ━━━━━━━━━━━━━━━━━━━━ 342s 1s/step - accuracy: 0.7531 - loss: 0.5208 - val_accuracy: 0.7445 - val_loss: 0.5338
